In [1]:
import json
import os
import sys
import time
from datetime import date, datetime

import numpy as np
import pandas as pd
import polars as pl
import pyarrow

In [2]:
data_2019 = "../../novus/matchingnemo/scratch/safegraph_data/Weekly Patterns/2019_Weekly_Patterns/"
example = os.listdir(data_2019)[2]

In [3]:
city = 'Bend'

In [4]:
cols_to_read = [
    "safegraph_place_id",
    "location_name",
    "street_address",
    "city",
    "region",
    "postal_code",
    "iso_country_code",
    "date_range_start",
    "raw_visit_counts",
    "raw_visitor_counts",
    "visits_by_day",
    "visits_by_each_hour",
    "poi_cbg",
    "visitor_home_cbgs",
    "visitor_daytime_cbgs",
    "visitor_country_of_origin",
    "distance_from_home",
    "median_dwell",
    "bucketed_dwell_times",
]
schema_overrides = {"date_range_start": pl.Datetime, "distance_from_home": pl.Int64}

In [5]:
read = (
    pl.scan_csv(os.path.join(data_2019, example), schema_overrides=schema_overrides)
    .select(cols_to_read)
    .with_columns(
        [
            pl.col("visits_by_each_hour").str.json_decode(pl.List(pl.Int64)),
            pl.col("visits_by_day").str.json_decode(pl.List(pl.Int64)),
        ]
    )
)

In [6]:
cols_to_select = [
    "safegraph_place_id",
    "city",
    "region",
    "date_range_start",
    "visits_by_each_hour",
]

In [7]:
read = (
    read.filter(
        pl.col("iso_country_code") == "US",
        pl.col("city") == city,
        pl.col("region") == "OR",
    )
    .select(cols_to_select)
    .with_columns(t=(pl.int_ranges(0, pl.col("visits_by_each_hour").list.len())))
    .explode(["t", "visits_by_each_hour"])
    .rename({"visits_by_each_hour": "visits"})
)

In [8]:
read = read.collect()

In [9]:
min(read['date_range_start'].dt.week().to_numpy())

29

In [10]:
read.head()

safegraph_place_id,city,region,date_range_start,visits,t
str,str,str,datetime[μs],i64,i64
"""sg:02a5f36eeeb941d29d47f60e567…","""Bend""","""OR""",2019-07-15 07:00:00,0,0
"""sg:02a5f36eeeb941d29d47f60e567…","""Bend""","""OR""",2019-07-15 07:00:00,0,1
"""sg:02a5f36eeeb941d29d47f60e567…","""Bend""","""OR""",2019-07-15 07:00:00,0,2
"""sg:02a5f36eeeb941d29d47f60e567…","""Bend""","""OR""",2019-07-15 07:00:00,0,3
"""sg:02a5f36eeeb941d29d47f60e567…","""Bend""","""OR""",2019-07-15 07:00:00,0,4


<b> Augment with FEMA data

In [11]:
# !pip install shapely geopandas

In [12]:
import geopandas as gpd
import pandas as pd
from shapely.geometry import Point

In [13]:
core_places_data_2019 = "../../novus/matchingnemo/scratch/ampnet_data/safegraph_POIs/2019.parquet"

In [14]:
gdf = gpd.read_file(r"../../novus/matchingnemo/scratch/safegraph_data/Weekly Patterns/Digital_Twins_Analysis/temporary_stash_very_heavy/entire_or_structures_clip.gpkg")
gdf_subset = gdf[
    [
        "BUILD_ID",
        "OCC_CLS",
        "PRIM_OCC",
        "SQMETERS",
        "SQFEET",
        "CENSUSCODE",
        "UUID",
        "geometry",
    ]
]

gdf_subset.columns = gdf_subset.columns.str.lower()
gdf_nonresidential = gdf_subset[gdf_subset["occ_cls"] != "Residential"]


places_data = pd.read_parquet(core_places_data_2019)
places_centroid = gpd.GeoDataFrame(
    places_data,
    geometry=gpd.points_from_xy(places_data.longitude, places_data.latitude),
    crs="EPSG:4326",
)

In [15]:
matches = gpd.sjoin(
    gdf_nonresidential, places_centroid, predicate="contains", how="left"
)

In [16]:
read_pd = read.to_pandas()
matches["safegraph_place_id"] = matches["safegraph_place_id"].astype(str)
read_pd["safegraph_place_id"] = read_pd["safegraph_place_id"].astype(str)
outfile = matches.merge(read_pd, on="safegraph_place_id", how="left")

In [17]:
cols_in_csv = [
    "build_id",
    "occ_cls",
    "prim_occ",
    "sqmeters",
    "sqfeet",
    "censuscode",
    "uuid",
    "safegraph_place_id",
    "date_range_start",
    "t",
    "visits"
]

agg_dict = {item: 'first' for item in cols_in_csv}
agg_dict['visits'] = 'sum'
del(agg_dict['t'])
del(agg_dict['safegraph_place_id'])
outfile = outfile.loc[:,cols_in_csv]

outfile2 = outfile.groupby(['safegraph_place_id', 't']).agg(agg_dict).reset_index()


In [18]:
outfile2

,safegraph_place_id,t,build_id,occ_cls,prim_occ,sqmeters,sqfeet,censuscode,uuid,date_range_start,visits
0,sg:0005a61c67624fb89f3565b58d1e94a1,0.0,2238338,Commercial,Professional/Technical Services,287.284424,3092.300781,41017001102,{f7fe75d6-f1df-4b2a-abfd-1eb38a5720a5},2019-07-15 07:00:00,0.0
1,sg:0005a61c67624fb89f3565b58d1e94a1,1.0,2238338,Commercial,Professional/Technical Services,287.284424,3092.300781,41017001102,{f7fe75d6-f1df-4b2a-abfd-1eb38a5720a5},2019-07-15 07:00:00,0.0
2,sg:0005a61c67624fb89f3565b58d1e94a1,2.0,2238338,Commercial,Professional/Technical Services,287.284424,3092.300781,41017001102,{f7fe75d6-f1df-4b2a-abfd-1eb38a5720a5},2019-07-15 07:00:00,0.0
3,sg:0005a61c67624fb89f3565b58d1e94a1,3.0,2238338,Commercial,Professional/Technical Services,287.284424,3092.300781,41017001102,{f7fe75d6-f1df-4b2a-abfd-1eb38a5720a5},2019-07-15 07:00:00,0.0
4,sg:0005a61c67624fb89f3565b58d1e94a1,4.0,2238338,Commercial,Professional/Technical Services,287.284424,3092.300781,41017001102,{f7fe75d6-f1df-4b2a-abfd-1eb38a5720a5},2019-07-15 07:00:00,0.0
...,...,...,...,...,...,...,...,...,...,...,...
258043,sg:fffe3c109b7a4947a7fb1d6043d33701,163.0,2226957,Commercial,Wholesale Trade,3667.824219,39480.093750,41017001601,{9a45802f-d4ce-4b4b-bf25-cbbbce4f812e},2019-07-15 07:00:00,0.0
258044,sg:fffe3c109b7a4947a7fb1d6043d33701,164.0,2226957,Commercial,Wholesale Trade,3667.824219,39480.093750,41017001601,{9a45802f-d4ce-4b4b-bf25-cbbbce4f812e},2019-07-15 07:00:00,0.0
258045,sg:fffe3c109b7a4947a7fb1d6043d33701,165.0,2226957,Commercial,Wholesale Trade,3667.824219,39480.093750,41017001601,{9a45802f-d4ce-4b4b-bf25-cbbbce4f812e},2019-07-15 07:00:00,0.0
258046,sg:fffe3c109b7a4947a7fb1d6043d33701,166.0,2226957,Commercial,Wholesale Trade,3667.824219,39480.093750,41017001601,{9a45802f-d4ce-4b4b-bf25-cbbbce4f812e},2019-07-15 07:00:00,0.0


In [19]:
# outfile.write_csv("../../novus/matchingnemo/scratch/ampnet_data/test.csv")